# L0b · Can the 7B learn to see damage? Training its vision encoder on building crops (GPU T4 ×2)

Task **L0b** from `Essential_task_delegation/Next_Steps_Roadmap.md` §7 (added 4 Nov 2026). Side experiment for **O2 / O3**.

**The question**
- Our 7B (`AbrarAlam/disasterm3-qwen2.5vl7b-l0s2-mergedFP`) was trained with its vision encoder frozen to match the DisasterM3 paper's Appendix B.3 recipe.
- Its main weakness looks like seeing rather than language: on Tamanna's **Part B** test of **300 true validation buildings** (100 Intact, 100 Damaged, 100 Destroyed from the 605 clean validation pairs, each marked with a red box in a 256 px window enlarged 2× to 512 px), our Siamese ResNet-18 grader was right on **212 of 300** while the frozen-vision fine-tuned 7B was right on **100 of 300** (calling 298 of the 300 buildings `Intact`) and the base 7B on **99 of 300**.
- A full training of the 7B with the vision encoder unfrozen across all six benchmark tasks would cost ~34 GPU hours. **L0b settles the question first on Part B's crop task** in ~1–1.5 GPU hours.

**Design (from Roadmap L0b, fixed before the run)**
1. **Starts from our kept 7B (`AbrarAlam/disasterm3-qwen2.5vl7b-l0s2-mergedFP`), held in 4-bit NF4,** with a new LoRA adapter (`rank 64, alpha 16, dropout 0.05`) attached to **both**:
   - the **28 language-model layers** (`q/k/v/o_proj`, `gate/up/down_proj` = 196 modules, `161,480,704` parameters); and
   - the **32 vision-encoder blocks** (`visual.blocks.*.mlp.{gate,up,down}_proj` + `visual.blocks.*.attn.{qkv,proj}`, matching July's ~28.9M vision MLP parameters plus vision attention).
2. **Training crops from the 5,798 training images only (never validation):**
   - True buildings (`>= 20 px`, 8-connected pieces of one class, identical to R16c and Part B) extracted from the training split.
   - No training image is excluded, and none needs to be: Part B's 300 validation buildings come from the 605 clean validation pairs, which already leave out E5's 40 leaked pairs (the validation pairs whose photos are also training photos). Cell 3 asserts that no image is in both the training and the validation crops.
   - Balanced by class: `N_TRAIN_PER_CLASS = 600` (`1,800` training buildings in all: 600 Intact, 600 Damaged, 600 Destroyed; at most 3 per class from one image where possible, seed 42).
   - Rendered identically to Part B: a 256 px window of the after-image enlarged 2× to 512 px (reaching Qwen2.5-VL at 504 px, 324 vision tokens), with a 2 px red box 4 px outside the building's bounding box, and Part B's exact `CLASS_PROMPT`. Loss is on the one-word answer tokens only (`Intact`, `Damaged`, or `Destroyed`).
   - Trained with DDP across both T4 GPUs: 1 epoch, global batch 32 (`1 × 16 grad_accum × 2 GPUs`, about 57 optimizer steps for 1,800 crops), `adamw_bnb_8bit`, lr `1e-4`, cosine schedule with 5% warmup.
3. **Scored on Part B's exact 300 validation buildings:**
   - Cell 3 rebuilds Part B's 300-building validation sample from `train_release.json` and verifies that all 300 rows match Tamanna's `part_b_sample.csv` (`100%` agreement).
   - Cell 3 also verifies the embedded `part_b_answers.csv` table from Tamanna's committed run (`grader`: **212 / 300** right; `finetuned` frozen-vision 7B: **100 / 300**; `base` 7B: **99 / 300**).
   - Cell 6 evaluates the trained `l0b_vision_lora` model on those same 300 crops (greedy decoding, both `CLASS_PROMPT` and `GRADE_PROMPT`), and runs **exact two-sided McNemar tests** against the grader (`212 / 300`) and the frozen-vision 7B (`100 / 300`).

**The decision rule (Roadmap L0b, fixed before the run)**
- **Worth a full 6-task run (~34 GPU h):** `l0b_vision_lora` is **not clearly below the grader** (it beats the grader, or exact McNemar `p >= 0.05` against the grader's 212 / 300).
- **Vision encoder learns, but grader keeps the grading (stop here):** `l0b_vision_lora` **clearly beats the frozen-vision 7B** (`p < 0.05`, more right than 100 / 300) **but stays clearly below the grader** (`p < 0.05`, fewer right than 212 / 300). Record that and stop: the 45 MB Siamese grader keeps the grading, and a 34-GPU-hour full run needs its own reason.
- **Encoder stays frozen (stop here):** `l0b_vision_lora` **does not clearly beat the frozen-vision 7B** (`p >= 0.05` or lower).

**Expectation (fixed before the run)**
- Above **100 / 300** (because the 7B now trains directly on balanced building damage classes with its vision encoder active, breaking the all-Intact habit),
- But clearly below **212 / 300** (because the Siamese grader sees both the before-image and after-image crops plus the exact building mask, while the 7B crop shows only the after-image with a red box).

**How to run on Kaggle** (any free account with GPU T4 ×2)
1. Settings: Accelerator **GPU T4 x2** (both GPUs are used), Internet **On**. No secret is required (`l0s2-mergedFP` is public).
2. Input: `abrarmohammedtanzim/disasterm3-mirror`.
3. Run All once with `SMOKE_TEST = True` in Cell 2 (24 training crops, 2 optimizer steps, 12 validation crops; ~10–15 min with the model download). Check peak GPU memory (< 15 GB) and that the loss falls. Then set `SMOKE_TEST = False` and **Save Version → Save & Run All (Commit)**. About 1–1.5 hours.
4. Outputs in `/kaggle/working/L0b/`: `config.json`, `train_summary.json`, `l0b_answers.csv`, `l0b_confusion.csv`, `l0b_recall.csv`, `l0b_paired.csv`, `l0b_recall_chart.png`, `README.md`.

In [1]:
# ── Cell 1: Environment: Kaggle's torch and transformers, plus current peft and bitsandbytes (as in L0 step 2), and no torchao ──
# Cell 6 puts the trained adapter on the fp16 7B, and peft then checks torchao. Kaggle's torchao 0.10.0 stopped the first
# smoke test there (19 Nov), as it stopped L0's merge (26 Oct): "Found an incompatible version of torchao". Nothing here uses it.
import importlib
import importlib.metadata as _md
import importlib.util
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "peft", "bitsandbytes"], check=True)

def _version(pkg):
    try:
        return _md.version(pkg)
    except _md.PackageNotFoundError:
        return "0"

importlib.invalidate_caches()
assert importlib.util.find_spec("torchao") is None, "torchao is still installed: restart the session and run again"
import torch
import transformers
VERSIONS = {p: _version(p) for p in ("torch", "transformers", "peft", "bitsandbytes", "accelerate")}
print(f"✓ {VERSIONS}, python {sys.version.split()[0]}; torchao: not installed")
print(f"  GPUs: {[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]}")
assert torch.cuda.device_count() >= 2, "Pick GPU T4 x2"

✓ {'torch': '2.11.0+cu130', 'transformers': '5.20.2', 'peft': '0.21.3', 'bitsandbytes': '0.50.2', 'accelerate': '1.15.0'}, python 3.13.16; torchao: not installed
  GPUs: ['Tesla T4', 'Tesla T4']


In [2]:
# ── Cell 2: Configuration, prompts (verbatim from Part B), and embedded Part B reference tables ──
import base64
import gzip
import io
import json
import os
import re
import time
from collections import Counter, defaultdict
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
from PIL import Image
from huggingface_hub import snapshot_download

SMOKE_TEST = False                  # True = 24 training crops (2 steps), 12 validation crops
START_MODEL = "AbrarAlam/disasterm3-qwen2.5vl7b-l0s2-mergedFP"
DATA_ROOT = Path("/kaggle/input/datasets/abrarmohammedtanzim/disasterm3-mirror/DisasterM3_Instruct")
MANIFEST_PATH = DATA_ROOT / "train_release.json"
WORK = Path("/kaggle/working/L0b")
CROP_DIR = Path("/tmp/l0b_crops")
WORK.mkdir(parents=True, exist_ok=True)
CROP_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
CLASSES = ["Intact", "Damaged", "Destroyed"]
MIN_AREA = 20                       # R16c and Part B's true-building rule: 8-connected piece >= 20 px
WINDOW, ZOOM = 256, 2               # 256 px native window enlarged 2x to 512 px (reaches Qwen at 504 px)
OUT_PX = WINDOW * ZOOM
BOX_PAD, BOX_THICK, CONTEXT = 4, 2, 16
N_VAL_PER_CLASS = 100               # Part B's 300 validation buildings (100 per class)
N_TRAIN_PER_CLASS = 8 if SMOKE_TEST else 600   # 1,800 balanced training buildings (600 per class)
CAPS = (3, 5, 10, None)
INCLUDE_VISION_ATTN = True          # True = LoRA on vision MLP (gate/up/down_proj) AND vision attn (qkv, proj)

# Part B's exact prompts (Tamanna_task_2/Plan_2/Part_B_ground_truth_test.ipynb Cell 7)
CLASS_PROMPT = (
    "This satellite image was taken after a disaster. A red box marks one building.\n"
    "How damaged is the building in the red box?\n"
    "- Intact: undisturbed; no sign of water, structural or roof damage, or burn marks.\n"
    "- Damaged: partly burnt, water or volcanic flow around it, roof elements missing, visible cracks, "
    "or partial wall or roof collapse.\n"
    "- Destroyed: scorched, completely collapsed, partly or completely covered with water or mud, "
    "or no longer present.\n"
    "Answer with one word: Intact, Damaged or Destroyed."
)
GRADE_PROMPT = (
    "This satellite image was taken after a disaster. A red box marks one building.\n"
    "Give the EMS-98 damage grade of the building in the red box, from Grade 1 (negligible to slight damage) "
    "to Grade 5 (destruction).\n"
    "Answer with the grade only, for example: Grade 3."
)
GRADE_TO_CLASS = {1: "Intact", 2: "Damaged", 3: "Damaged", 4: "Destroyed", 5: "Destroyed"}

# E5's 40 leaked validation pairs
LEAK40 = {f"socal_fire_{n:08d}_post_disaster.png" for n in
          (81, 188, 218, 220, 222, 235, 301, 341, 348, 396, 436, 455, 497, 504, 614, 1188)} | \
         {f"woolsey_fire_{n:08d}_post_disaster.png" for n in
          (9, 30, 32, 137, 272, 399, 435, 474, 498, 514, 529, 573, 576, 589, 623, 632, 634, 653, 679, 703, 722, 758,
           783, 810)}
assert len(LEAK40) == 40
assert MANIFEST_PATH.exists(), "Attach abrarmohammedtanzim/disasterm3-mirror"

# Embedded Part B sample (300 rows) and answers (900 rows: grader, finetuned, base) from Tamanna's committed run
PB_SAMPLE_B64 = "H4sIAAAAAAAC/7Vd2W4cR7J9H2D+pHCR+/I6mJf7FURb6pEIU2wNF9P6+zknsjKrSFZS7FoM2t0i2wzlEtuJE1G3T+cfw+2P07fzcP7rfP80PF6eH76cy7duXsbX78PTw/P55svd6fFx+Hl7/nK+uf06/D38Gl6G78Pp4Xy6+fn3cHe5/3Z+fOLbl9v7r5cXvPvnP9Tw/fnh4fbL6f588+P09PT9/HKj+I924ebnBZ//evt4enw6P/zfz/tv7z88/P2vfw9aGVf+829IeLj8On8dzJDNEK0atBuMwpeSFx/++Q+NZXw53d385/bhXIQZZRaETZ/qStF2cDEPGXKcpQCfsryjHDP7634/Pfx1/jUuzcQPl1Y++1bmiXv9lTKysYPNdghu4PqST3wrIu3s1/zn7vJwvv8yLtGFj4XWT78R+//3T6cvT4MLWJWHGMixg86QH/hOhLrh5+Xh6fkbNuvl9u5r21adlF6Q+e7D3d21g/dxcApC9YCfGC5zXKnvbK5btblNZBqSHkJ2XHFI2FzsN15FZOiItFqtP09IC2ZI1gza8DppnfgiAmNPOUxepRyjSIsLa/yQ/WDiYDVucOCLiEzz33L75fvpfFdEeq0/Flk+vHx9tFZDggamjK0MXKnFv3gnMnNvmdaZDcv0uLJxyAlawks0pJT4UkyA6q3TuLhinTOLoPEVB49/8dZypZ4vRawenp4f/sTFOZ8enr7/9/n059kWcSElkfTu58MFa3xY2NMI8xZwcxJNHG9OdGLsRJAZfp7unm+eHp/vTz9uq91Z2s755/qrwkUJg88iB1/Yx3pJdd/mwIhvsDm8NDnowcPwQLiDcO/ktch1w/35Jw303eXy9fb+23hPnV+Q+fqTPb3AZuLIYG5wZLBxGmZAV2l+eLlc7h5xNJPbiH5pffPPdTfUDRkWPIVIE+7grKLnS5EV4Cxfzg83P3EoN39d7rBZl3FH85IWvv9036TSnHlxjAF7yQXS0o2aiGP+fno53d5Ohlyk5ezL0b3+4burOVsfVcDgxli6Q6ehh6E5Rt21Miat0r7xAGnGLLeRfgL657NvnlHnnvXOboP1VmI3Bw8T6rUoSEhc9hgBqHehRvT2mlCj+t8Bau40XdOQIZJWLY8mG76xs5vK2Q27CQWLFpEGdo5qB7Odc+RrkWrem6rxbvpoP2nKZuEaNw1xjCyumBvFd0XWB2FN9GtMzCSYVoUXtDhfeuJm2cyyhTHKrbYw0HVNmw37ksS2qNRMjPG9c3Q2rfe9sKAJltQjYINSQhO18/T5o9CwXzAch0S1xxGKN0LQ5uRF5HRjGajspkAfVjRxiRI/IeT3/G+RmXoy1W+C4Y8DC5oW7GiMmm8ZQ9Fy14Xm4ccFd+7m6fJwf/p6qdnMUoj46oN93zsED3cLNc/i6RlA1XAf9uX5/uvptZ9Xixf01Qd7HhcbqeBtcx4gkpGFtvK2SNP9M9wSj+LOwL/j2kmGxqQmtDDfmn7E5NR1ERN+e6R/gA2NMDUIzQxsdRyje5i6xxM+ePNweTy9TmQQVSwp4MLHl/eVUWDEEXJPA7c1wtTZKtctXRhl7eoLkxBNYF24MPjdTCxw3/m2iOslTsb79X7Q4Y5CAZgeGokUYXXs6Cnw+uP26wtz/9eG1KalXX372c5VNZJ4x5KeWRoaZsR1T+Nbq6bhT1b4XXglxEawnuNGIoXwdSO7Qczyuj5prnENHZxCVCLU05bCcjehvSjGhBVp7yQywYgBRrDMuBHCMJTxY3Kvetm9V2pFdl9lIkyzxfUaqkIIU/zrdNfd66y3ZBQ4TtxPwxwNy0R6wb/CKNT01hlXrXOORKVcACh6eojH7TR1qUclT4ylNFEMyXk97m5GJOyrWNe3dcZusnWaET8kZ5d4h6g1urwtkv0HVjbHTZLhQBy8V4J0GHeqTixvi+TQ01dt7aaUPzE8Jx4G+dj2JC9FZuyG5htlIhjIpmbjfJNb2OpSX31UXhUttwTZwpjTdwUebLSmGXlk5f0bZdad6ysVwuHC+CaqLeRrpiNq3GWvdj7ZmgAh9CFIRrXNtE4Jd6kaKVyvxeTArIYfYAsSEAePrYRBtAxSNF+KONO9Sb/x27+7SZDIc2TkjrgWyQnukR+DWZjnZaOobAjrjT82E+ICLjB8jiPeiWzI19N0w5+nu/P387SzIghRTEHI3vz0g6SS0TKWYBk+F5RMIN1Rjt83gSWSA53IvoTqOL7IpZkxS/dd9Di51Z6bljUa1iCs6CRsgQIKWJUSx3iUmUeS5xOFRl5RYnMKK63YnD8I4IGzDpq+1FMjId9YZeVVpOb+iYYr0VXmeLQvwRI9RsoVgV/ZUS0ATWwXNLduwUhmgIUgG3GsIdU1IY5+D+SqRYzzAyC3Bl1wiQUmBmzFxThqhhpvaFhItcbIx/lwfaoFBQu4HgzseGRwi2EMAoLtVcZ03ILFwULjVhrBOUwJnCvOgeCn5xStd1ucIiMNJxmXKQksQ53RSQR/UHhHi01fSFy8pCSEBWqWgJSlq/fLqfqn9R73kbCOwZG6ovcOUGTVexz3UtnBry070MvDT0jRyGpBB2uhCpb8LXAVnF6R4kEjAAiwnsFICmUin7P8oYjJu2PxrTxlNFEAljXfVgCi6gOe20I4eAaAuAZWE9lz4CIBO8bx+KLeOcioKuIFzpWaHzEl6EfguyLU7I+7zNBB2nFd6uXiOYpZHxEtuM2dEV5eHdg86gZ9FTwWTeB4Z6PrV1fDhv1FzgN7AJuAA2XMEXDEodaQfbeI7Pwm4FWXyCozqbTyFSZOAOohPRfpkr7OmeDqJF4f3JiScSSJPEZBcRFvNflK+KxaA1pUihCQdyBUJxdWRKUeRBCyXR8NaxbHokCTDBbxZf1Qw8bYLY6bZDcSZJzUqEsWKeWkCisnYD639+eHXzd/PD9+n9TRLZbK3ny0y3PwArpmSTMY/hsCQNU/J90Lj/2GkAAbmRgaw1UGqQoi+ImtJJhM18oa7Td4Z15NFM9wdixd2xLejbFPsvvChlMYgqADWuJJC6BrMfIqIpfrWDama63cRD0wmbCWXBx+KTVdHn8Ma8VA/bMpflJLCURreVek9tIrlzaUd0OQJDwaQe2lugu5tboLpV0Gto3fAGzD2MF1RCWMJ0f3xaphVZP03myO68N1u86sIpLKCG8spBH0RUjgiW/zTZHVw5qVz+szVgQeiOoivSN2VIrKueH2uUsHVL8rgX7MeMJFxTlmRgAlmIxTAJK7UQ+S6Q0aWaydLmgvtYMo8Bjg5S6cg9rb+uIBvEZgGc0WwAMWHYdYAQ9o677BOVZjBtlRZAM5TuwqoFi7ZZK4jAhLUeWJ4/dNxEbm6YMQ1y+cmw2VGM0gClbbyVVh4Vw0cXTMyC0XQ0er14aOhFJwUTMdsRY74FvokePOfLV2R22QJJnQAJmruSVYyLp2ZTxB/cj/gVsqmLxhCX30GTn3PbE1Gzwxzo0yuCZxkATKqg7Sab1NIZOy67gPODe4hYBfiTsCNSRTzTayoe7BqHExuvls4BgFg4MKS4me7jG2Yj3RkLc1ULUMxP8uQSbbF+YaiDfDGRbRdQtn6C12tWQNDYcPDHQWim8dCTNBXkex7t3ZZb2WxA3rjMQpNwqJbRQSrbo8Y6+28IzpY0FOxeXEgRHm0M7Pzi4scgJUuJYTUGMn2hPcQi/ZhaANKJk3vqGKh+FFWjB41l0V2YZ0+K68jqJTb39V2ESSZ1HDQzAJcwTJsMbchOauw/B2Q+ke2QaOMCEsRdVB6o/kAlbSo1bH0Gfo3wUYl4MlVG3tdLZLlOMaMfp8NVZNV4zNpHvCMVoyySrBUu/BCGxenxk/+b/CpkY0nOXNKMnuzbEykp1qU4qpJF2YyQZod0yvARbJohfAPgBEtAEBZqhiRUxulgIMF/wG9qGV9EIqGYSLsms1Da3DMWw5wkT0UJDqRRuwy3mmFXFfpjrLcLykrEAUGLca1T7r2K7qbZjF+TlJ9WEs26rUyrYa97aTIy4Wwn6TI84vrM0kjpc0hrQ50xIaotj7Z24ILxLNKjNGKa+grD+ZGaN39ZMN2FRNKYk1uKnrwJhDakiwOmzHicJ+ZNThIbRFHcZ2V6nXCyWrGgGWT8LSiwxHkBDHagg6NGRkUmsNAXE/xEswP66Cf3bWOuLfh8U6rgytuIlsxUtifRCAZzV1w5geMxBI43oABXE4M41MEoOmUniiJ0034vDHCbHL+e+fd5fH28t9EWSKg3r9o/d+sIGZJMKUdB6/P6rZ7++zb5b5TZ8u3eggRGP2UI3EKlLofLubefGawHNefU3m1ONkBYQqmBuBjBpEWXUE4kamGus1PkWJMchLzA3+AlEEMcQjrt2Foefr6HixRPX+0118mgC8EgyD/Rxm6hWz5oiVEkggyRMJFAyAJHGalUdTD9TaY2hNLHC4wpZLxR/D8DR/bHek33A1SZyUHmEoMzVUWd/nxpmNjCYwWIGjeGk4gkOGmWutR7Yb5hgdN3UbCt5P6rqw10lpMk1X4rJumrSaLIYQCkkUjYEueBH1s7oM+K6dyWkwM2zuYG4DzRSLwApA08vcp+n7axsbk7TEkuJHH4zof8bjYOfhzrgbcxgYUmmSEWgKVkY32qj+gI1szSbmCMNUm4oLQaoDoQ0AcOaYSrGE/1AILaxgGHWXZy4EStKvLnwWt30Vu0UJK7ClccK+Gcb1mv6D21TTBDkVS4tWUA7pGZ8q0gRYdljdZN18YYdKng/dkz5cRbccq9q7fkqVN9kaXQAUQflJaUiz4QbQlE6VWnu9AWyEBIQ9STaU/XlseGpbm3bqcxSVMIWkaQtTvslYjm+8diurmIgPGeezQMzcNBSuRkOmvNqvM85LjxpT4BIxsnraIkavj+hMlUqeYt+mFzXM0p0ztTObZdrEljY1GyWAinS2Os6qliThHoMtlFAmMJiyNJ+h1WeQuPWitmi2TYjQoxO0kiAaN4Olfb/4HbbRNXBwKQuvgHyRIU20FFzbvmEL7nrDRo4NJJArzV54Rbp2Lm9GgXGx7VCvbDt0hU7AShTvJ812bgQY7VO3Yc1sUA/kUuwclbZtK44w8e0os5NLWbWWMSFtMLg1RjobydaaRpkQdHjHZTRqnamBd8dZjfQltjiZ6aIE3a/VbKkiIsTGcmQOjZJMIiLLD9UhLfGKxzg0ZnNlbMg/J071yFIv5cFpeTvKsgcgCuzm4d0vMQWsaC7FjFGk+yCsuDL0DV52MdG6ZKkgOBemaxn8Ydkvlig9Lmx5kalCIaRGdMEf3q9hvDB5VRHB6LEltgygkZaeKmuR0KeuJvS96oeFnofSessUNBFta8M2+r0KvyGG/5b0yhyCbkmmN0APbGNok3LcO0qz2G/426OsUQ2JZ9KE74RFyKytuaeo9r5AjfGWmFdkMm3o+NlcXQO32G8bV9sYqEwNofwyqkXwqZCnCCeave+RkZliyUgbPi/VhJlG27Vx2VxJd9WEEgAKsSPKMxRmilaDxXhg5ybnQgnPxklCqmOclaTjnqgw7iPLT0kG++Ar6NQIvUyf9u6sYzcKB9LoVm1nj3pbWjyEjFUcPlJS8haSlEx8UK03FMXhHXcUxyXtikr0gG5x4kbqmI8oW3DqBqrqbOXzYldB6ZndVOSkO80SktEFbIxMhR0tVcyWF+Kv0XH5i5DQZ8cLAEBUNKWmtmekqfdTv2IO72rSymS0LDxwGRETpkg42X2o7vO8NNNHkURErntyU6yYeumTclum3TFDjDGO1SBfGolHiR+0hi+zwK9pBCHyxUp7UUYjBEI7KWMK+3aDMhEl8sw2fAngCAa12n6Kx5CmGJ4ydiyzYgqDsFo5AALdTqZFJvhvO5kmvFbm+sEVG2HXOTtXzzz8cX64fX56W3HzxS2+/WHPK8L/sAUzQIoVhUSZaEZhymq/KFyYCkq6zcjpo8kxaTq83I1nlNsEZngZX+DGuqWX/oxWDcqHNYDTwrEBvFQuGI3nqXKR7VFTEyLjVPYPFwzTSj2sSnX9kZBuI4zCmXvOp1GsUW4m1n8wnjFeGcWRt1jaB2SqZ5nbVLUih4/GYOitTW+E/pywwiQcBldkfpN2ZtzgD0KySyICBt3H2f1ZrkSpsBZJseOZGYEX2dLjU5pN+ct7R/2kn5FUF2V+IVENCqnoBku4/ZG7dgN1mexQKwW3EgrMQTFC8cc02QTJldmiVb6YuNbJZuoDgl++ktZP0F0z/JCBwjIDMzbCNNtTD+wJZaBTJtNlGWPibAtCeLXubv/4dSq3sAhKZW2z7/eJIUgLHTEiLYME2eXiVJQ34+/3e0XFUj9Q4qmMFNfIsjNtHaE/zshsHMrMeJFkJZbVmXd433ANo7rVJ/wfa+ROjBvysIScwWSR3qJOogR6dBR9mRaGIastzZEhNpzK9EnE2+iEsKZkgXFrqXwm+8ZEMbuPLa6pnJLWE45syFJQJCTfhO5ta+YdxJmReC7tSr6Ym1Tl9iZyqbzqCrfhM4HoShCkk+WUzOtcZ3Dq5Y5wp9b6KykvEF3VLE3JoBSr2sAEzlLoDMdTehs/1LGxh0QUsbAhmjYCkHFBX2Gi3qQwQoVjXzhXy8BnRr4xutelCdBnNTGVdRzOOpRZeZqN75Fjo3XTVB2PGlCvJSwgvkJrzHA2tbqu6fONN46lYhmCVlDgEFsGWtq2xfmDQSdhQzzCDgNUzI2t49Q5FLjaCKOO6WrGWmkN2bbl3DjbILWsnUrVR9HzqoLI3DiREpSzq5CanSA1szT+eAOZhPyjGIUrCx8jU8ACZ+hVafaYGgUNMCe6SSDNXQ3wBKFJdTvH7qIkTkAtCbhwpmYKvYzft4zNUqSWUatkO5C7HTmhS/smMBxDISc5lxV6qU8ShyUhv0WApmeNQGXZMEVCyrwSmEndkGiab2qSjpufzaIWJw+wHio20Fk92UCTd+hXf8VH5rRXU6x76bpgt14z9faDyVzXPmDBciqQ41NkjBDJwAiyZmwHFllHGh9EgTLiUBUSHWtbYcrErDlqqAsCQI6yl/Fn/GJXVDXy1u49qZgQKSlYzF5GVumM7W2s6+4wGge31LsZF9FlFr9iObl86mQz1u+L8E2QUOaKTZkLkmQKQh0Dbexy2csrvdLaJuGXMyTx5TEIirTWtsYtfRDV5hTV8wTS5dhQNslTbGv7iLNeZAde8RyLKLRdxHppkBaotov5KORQmmbgk1UoaN6rSfNOHTZlyUhneZTx04PA0E0hnT6muSxJkUY0nxFCnj0RwSyPkllfm6Vb1jJxfqzhuzA9EcHZA8ZdkyzEGQQ4PysFbs2nTPm2qW7v5lLWflXjsxMF9nZ6jobzx5CEdaGZJlWnSo0vRWjYkektU8OVPMNKcCCytNQECbm4c9/zPERn20M2JT4ny3UGV7p0JIgYyDOHt0xJjLmXMn+dikCsdslVxrByKhkHrLA9UJqw+KANPdker/aCLKY5z+z04lMnCGASsogc11vl6X3CgPl4Ccdgjkwl4j8+TBneB3OP3cYkOkijjnCyBO7SEz3L+GUYyCe/tmyRxU1GmY4uRpw9HrWmSHrD3l0tQQp5KckkQplEPKt3G/9BspX8lQaAk3IUJ7pZis3UAwLhVRnwvR0GD02Dh21kaiyXk2Ny6fPruFwO1VguN9t0VbmZTX+RdMwocSkRNGRULTL1aecushm8jqf9ICuXTjkWf4yfPSEo7/RcC3b8pjKGR0xnylPpYGm68XhWMPHX5VCJML2Xqn0qj+XyDLabqG7rgzLrR4wZeXIbkbkyWsFwekSs/j2Yw9BAK/Ois5OykiV1ILRJAPxJN5lJW7iXZbgLWzlVYdYN7F+rBAw+M3NB+2JepX2OgArnVOWxQGjC7BlPYecnWE0lEXJLhCRQBjvPhtdw6OhhT8sjYVAXKg/vLvtZw1RsNstU5RWhYrWjWpJDxvU2lNG1TrcJHSbsOytngnDY4CFtA/K0VfZC6PYIABPyvrDc+HgQfBMZBufWcpxLswix97gZEPg3FH3kiblsly0DpDmAyE+lvP7cYx39egJo5AMzxAIKz70UL0eJpgOt5tUcV5wc6FFi5oR+luJMnO0+unZDDw0BXD6byMg4F6InKkyd5Myx9m4xs8Jk8Vke7yiPDHTTg9ei37Xhqz6cjF6Yz3NIpTWw9Am3je0+28rYFRl4K3bLuKooTwKW1vmJ1276bOXl0UCfvaxanl3N59Ya6ZoPNk0PYCRXec8HzQmzBa5Sl/Njhjx1RKJ9aL8ZBAJk2iyPWZRZZHrayqQOq2wDtWGtgZOyBLhhCWB6LqHuZ6Z68RA/P5+fvPYyF7DQhtjK19L+ZA4DVLWUIRmtWiFKZ6UnH42/zWE+2pS0OPhUn4Uk87uq5E5iFVcnx6WbiCOlGUEzDhKuaJXnD8MfB5m2WkYfkyHVapGp32autzxA2xTOopaKb2BcwLJEjaNfcZhfmVudtzzEIwv44Dn82EmizL7Qliin7gx2v+iqP88vIk+LPlPqWC7NCvndQchbHt1DegtZWvK8TaH65DJBbHyKqNoxavfkLTJAkOIuy1lsFIlVO3MPRkZH0BYYGdGsytJCUUaUCOpRZZojmt5lQJjMlB7KA0o4kWEqSOw9ATmIp5YW2yQ1V44rTG2Jnbldi0OlP1U+J86aZUaZlevpbZrdGL8jpCuzJNlqa4U+7KUO2tDNHBaevbJm3AN3DYlG4NQ1Ky1ENuv26Arzaurx27q12jJ6JcsQWylVy7OQOQalgTs57TrvaQbtkJIVJFSWyWsqT4hjPqRHS8sD7dmIIk/TljnvKbeJL/8DHgJkzeCFAAA="
PB_ANSWERS_B64 = "H4sIAAAAAAAC/72d224cuRGG7wPkHfIAg4Dnbt4vEOQmryCM7VlbWFnjzEjr+O3T1rTXlobF5v+zOHuxXmvtv3hqnqpY3+n5cXf/dPi8u/+8/3jYPZ2eD3fvH/bn8+7l33f7x/PXw2n358Pn9ccfT/sPh19//PKDHz8+X/7U3//28vvTzuw+PZ9O9+/3j4e7z/unp0+Hr3fm+z82pLsvx/PT3Yf78/78dDj988vjx91vh/PT6fjt8GH32/57eX7+utv9pWl35+P7/cPd7/enw0XMGVcXK/zXL4Lul0J+2p/+PHxby+imkuybkhVK6H8R/P3heDo8vl9LGlJJ8t+PT/v3T29++UUv7L4cT0/PH5dKf71/+PBXxe1sbE/Fo1DxUK54S+ckQdJbw7XlJI0glxOnOP+qeP/+0/7wcFGM1hKdk6UC+uC4AlojldAFumOs3T09n/5Y+uOwPz19+u/z/o+Dv4ileW6qqXW7L/uH57un8/Pj/vP9j4+k69uz8reyzBFEd9iwezx8+T49PByPH+4fP659GyLZGXH39Xh8OC8N93PCmWLoqnTaPRyX+fPuy9KQd38eH5baH9dK564v2i6fy/7r/v7+50TxopZzBIXEr8TN5KxoszQ35MApOnO1FEzRE4PGWam2JvhK2WRBd/21rR0bJ/+mK2SVykIyxeo4+c/x8bB7pVX+LpwJVP2i1GDBz0wHJPVFXVw4vPXsJOpmSdSkiWrHvPt8XPr17ul4etx/OP7YHJmeqvvls3h+/LB/PVEbw0yn3srNSK6/3skrUTBtpfK783756d3peN6/3hMtUz0zAH0odYTxvqsjpA2Wi7XV6Orb9Wn3+f7D18XEm6/Xz1Rdp7cfm/XJMELiIkEWTFohXG3bXBlqwUg752iYCgcrTsg2VzaPG+MkOKmYk6nPBHJJtfdVIcifnKPm/BArH3GmzklJGo+270MOk7g/8PRiEmZ5MJnMza0hV3pp61S3cVg0YONulzba8p7EkXv16MRuirGr7l74QI1PiRioMez+2D8cPh1+1vxFxBozb23kYgS2llv1Eo/rMzNHxEl7jojkMUQWzHLrpbZjcDKbEs0dkGzhSG2KZ9XNYhW2VeviEWJqk/DSTZilGjsFcYbzMSA7oBSVl7OU5KFa3tluKk7F25GueSfNV8eiFJhLqpTRG4cfs2y9gJORT6jVNUws6WTBObzhCtGhp4XmDpq85ql6CvLVZKIEo3h7GiK7b5mSOP+F2TYNx2kqHntdJob2NEub55Q9vSWfxGtdN3dtJeflSHL/eDh9u3v3fP70cyCGTA2Z2UoreCSv6WYnftLOMgvT7LFT4nady1dZfpqpQ+Ic9V0M0r4qzORV5zwJVwCOuu6Y5+tveC3ZNOU2CemwbiLzGWfRVWj8zLVZFpcSa6j9TBbPFz5Tdfb6W4YcWreBW0LyFa/jPrScyoult1RnTKCXrGHAzPq+npzl6dQ7xsFlrr0es+lak6yx0vl2ihNVRnd1xWgcp+Sxb67JvxquWjBbx120WSN60aOhnbUmFa+jTWJrPGkfd6yZpXrzpczilEO59aw1pPdi43spOdN/LJ8xNw8du+0l3K6iH+BBsjboh4bYxZVemv9Disz8b21SdsYttxxEgMPVXYWVfefe2p5Rl6U94Mx3ijOq2zjrLDYdSjc+S7NTl1HVunqxbJaVLJ9Flq0WKxivV3fbdXFvneQ+XA6dVBdPu3f7ZS05/O/Lw/F8f3y8iDgPjWYnuz/K3p+m1svF7li8c/QH4g11lpObz39fPM5LFx+/r8WvF3j2ist6p11Kr+7usb7d7bEZNhZlj1xnQJq4rLjqxHM9gy1+9uJgdDPXI7Out8z6LAdixMaQwMXDrnimsxUHu3fcQA6Ou1SttdziXZevT1JrwYIYHJ36wgtD3C5eu5i8zcrY97B40YWbWhstFeo5K8X/2VBeNyIXghqNdjTb0kDawZN28ZYXL+H7Tg7Ra+/Ko3gWmRy9qkf50jd5cL8fk/y5pYC231QM3iMH4iwGoLGx6VHYYXkmnsqm6y8ldQ7AZOULI3ZbWfKtr6vulF1jVb3uLjyFyiTfuHqnOGArmgqfw9rw+epyZLM3i55C43LfGJHDSiK3y0hZbElHhRouoX9g32wryrG8Jva9OnAjemny4leXnW2ss3rYop30T+hTUt7aT5Oup8pOs36ls/ptzOJc13r+YqXZ2lNb1VdeddVvb/btsRS1XcwsbbNMMGx3VCJ8hfiE9monvTBIO0+ck6nQirMcaDVR083iZ393ON0/P729b4uNU2A2/RuFLC4dJnC7+jwqVtdmPygK3OYgv/XsO7HnWHmdOTX2UarF0/c9K8yMZ6RS1PJVlqFeodicB+w+ll1b5RU39cjQWPVAo+VaQZ4Gc2oslx8WH7kEQu4e7t992196+iI1N96Que+udZXl3C3+dPGFTd8zQyNeasWZitl0i2Nd2VfvZMf6hkeyMvLgV+qbFbfo99H0ZNNKj6tMNo56K1yO+Q3UMc8tnnbhbd9GboeKZG3bNVmqlFJA43J0ofSmQSktnOx8r/utKmXNlVcMVO2d4SJia8Pc2coFRN9KWHrMzroTnPPc1Uu17kFxO7EcuvWuc51Luk9qlkd80nQWLbUrqfjht5IMVPskb0YYN2VAqDy2akxb4jz/aVynUfBuwJMK5/2A5+fOB7HiS/ge+bH5iB0Ft7unfPkVDbVS+WpwyvZfl8/vNnb2Rh50enTB0GNyS9mSIWeF/COu/GrDUdF/LnjlXAEuhAGRlEsHcO7zaklTf/iBCxMYG9te5XnceTKUz/rcLjQaYCff0jGLl755Gt8uneOSElULWD67xDlSJQyawT8uVjZec2wVSc1varbrNwlXr75x5xFnMJasMn8ujnedrC2u9IB9bZ/Zu0YNMR6FO1QnN+pAmLy8CaoN++seWDzuhaE1Zaq+YAKtlo87DUsr58pueMt6hlwCH480baJTVjy4TVLqoCVcgxrh8sN2O1Gz0+SEUy/5LMpNXkze6VjJMCDGzE0RiwhrKaiYYstxt0ayB7783mW7u2e9rG5u8b23RmLLIrMZcV+7eN7F3aPlmm525Dm4IumHzbXC8/ZlRiLHtvzAvffgOcuxwfSU8coX/+rjtuzENospIuIU+QdZTnwCz6VPcounXm2DkaWT+xL2D+48s1NP3Dri5bvLwlssQ11L5ahwyM6pkMCHOg2+eu7+9pqWnRnyjL0aaprAs15c1+9LppSn58fld3Qi+dct+q/vZfyHXXLIX37wqwk8r3xJOy5Z1f/6E7/Kc1nm24uvknS+ZoDOQt9ugk1I326By0/frk+mq4dGkkYC+1odyIz2UB34HPfAB02mvK9quu5AqKq8StrWrbZnUuQDdWAy5mMV4C86MDvN2fQxWZWklltGuGz77d3cl3y/qqySjf9iwS/r4vr/X5ng8/NvLfN8zn6ghfqT+V/k3bItKTWPRnr/agXS4M0Wnf0fM6PCA6g2FAUIgCrRjQuo7kpJfgBWA5IosCWrQxmoNg+FHcBqwUEIMBsKgIJqM3URC7YKr0ExqK42HNcAOJlpYA6qBkjuAWBBCYGw0ddBcX8tdLYSKKHaWDrkhKoJGqWA9QcNVsDMsKgFoME6yAtQXVgOAzCEKS4Dok+//cJaSoPaUK9IIN1o1UuwOOpsooF7qJZ8Gj71RcWzs2CBJERstD1EjcCUOxkS1etSEipR1dSgTFQNsNgJrNnj8K2EEqSiboPBVmAN1QWxqHY0S7WAyq/EuKj2Aku9ABwTHRAMrLm80t2UbEEFk1FtLJqbgVWEJGlsyfbSNapNw+E2sArQ8A3Mj6aA4tgyweE52rtDh9ZRtaCC79hqJwLpAVQh3sIrmxQysG7ZUKCAVBuKxIJsFVsBFbJhgsWHAGu0Dk6k6njX4ItUDXilzZ4gH/RPCySGBDDQTyWpyk8aaRKqFmYlf7sgrwIwqQYMEEQTMNhEg29Sr0IX8KQuTRJQgKmtn4iy2QMsJQU0Q3BTQAuT4plc6nEOrYJY0CCt1GOvzA0cvz0glnrpSTLLZnn9cHc+z25B2qcf5VLXT3qBFdKcZyelUL5KV5AkGKgaLBUGCXo0wzfwLDgGqYUbfPHMAmYQCwxvBuwIPL8taECBR1NvpCZADVhoFleDlJuj1yDhxWb8JQUPuEEq4tQqIlnww/3sCBAHKTnNxkGMkJgcxEQ/P2dzqM5jIx1oxk61YRSgO3V9HQpP3YYb7cThKT11VRrbAz7SQCA+oDSJ9AGtaOB+6l0xj3qCoAAEqj+RMSrx75K6VXv7IXcvhw8CbXi9Y2DFCgsYQrqE5g0hRhAQEdhGvViiTQMcqgg00g0vqrY/QTMC372RbCOkDiTqqC7qB5/4WBhSXTQqnmOkDSJESwJHC8VOAm2okJTq3aCCVqqa0GEt1U3Q8CXEiNPqcckAiWSqi45/McFAm8CnwknpJFkxMamFB0izEYF5AuuQR9+QtjOgEFEFHFTdgNOcHipPzylaFFIRDiOFWOigSoGNlUa9xKCJU0g7qaCo6iYYNtVm4gLTs5er6KowrDat3OZtUwflCrRDM69AOyQBa1O3g4oFWprGpobph2bV134Oo4XlS9CBam1asTeI96XRW5u6/kYvNXA4Vz0fTNTfeHWAuxAjKgyvuolZMWys0qNZL+BEtqKD/qqnAKJZYIgRDTRY3UI3K6yexkiJHrbZ4TpEsXpTKSDG6gYmzQSGlcaa9QIHpKqoIMm2KsJiyoCKdFDLwLq4Ud5UmmgGWgk6u7+KhdjpPZL6OA0Ou9NhotUtzLfI6tbBTNvMSkdy1Krl9bf6ennoGmiHw7C9yVy8aSWopKOvd40Gpa1uoRvbttlOOMptU5LHu4GG8k2uTTrQb6AdOzzKXIMQV9f3WmnYKu3EQeTAGYSFyiGNlQYEz3VA58BumG91h9INpNsyQEHqgC7pZtbVT6gsxQ6y0U212+yEoBRxK1aBZd9tlryTh7dRbIaQtyFJM/OQQZ/7E4pKmY1Jul5dVAG3tzVSeAQfUhGSxgdWpZPUV68Cie5DTNwowT3F9QMtsKQ/0EweFfqqQwKsdrcKGrBugWEFIvocOhDsiKAVcS7VguMKIhY0MIOb7aSBHqxXYx6YMp6HE1ZVeVohYkQHXli34cZfH3XgDRErDOkQBDTE29zDsCREZGdMkhHBipCsRNBK1spELNvopSnWsRwsXhGsgxtN8VFLCVapRDeXsV6D2HVTVSl3F7yx/i3TOEewCizgEWn/MSHd7/bngy728U0zvRjAoY9Nd7Uv2hzxsV1fkfhYkg8DwyVeDLC0x/YW0qM9ltRJ1mN78fVAj684Xy/aJOSxvfC6hMeihWa+I1BqNyYp1EWcpM8AxddCOxbFGbAjUHb+0gsw0ox0BDRJJgVgIasQuCtLFQhzBJRJmCNgAWE5ArK6IMeiCQbjCNRAj+JYlE8j9040wBGwoUhvvF5gOXIjsPvDsY2AOIltBCw0MxsBTU1gY9FAGBYfdtEfxmq8yKthGq+H+wBE40WXozOW2zwsh5orA3lQVNDlEKWHZizKK4IZi/o0lrF90Ifhu2JVJGPRgCaQsTRIeRwj0EojWYwXC6MwjBf1DgJjeyux+EXAAkNfBOTpB8qADY68CBjoS5xZlIxDjhEccxHQnwbPbXH46baduAjc/CG4RUAWZi22b0TaWYtAefVYi0X5gaTFi4E4em9AMxYBEwxiEZBX4CsWdVm6YnvRVdGKRQMkVxGoQgdXEbDih94UsURFwMJIouLFQjNMESi1DkmxKM1xFIGyj4QoXtxTFD8RcH/pwROL8oPQiRdxkpoItI4SMrGoPQ6YeNEfxkq8yKthEovqzZBEoDMpQmK7PotHBCwo0hGL+npsxIIzfFAQ1EW8i4pYVOSYiO1HETUmYrH0ikTEov481E2tyUMsx03gNEQkKmMMCnEVBymISLlJCiJiAocgIuoDCYirBQJ+iFRAk3xYmnU0uYflGuhRD1/Fda7RSOOIh6sBBHYIdGw77BAR9WPd37qYw7IFBnJYrsH1loNGHCJNNI0MZmPhhkgFNNmG5XA/M3bjrYk1LOu7oTe8mkjDsv4goOGqjqd9R/pWDWRYltfCGK5ymgTDsgU9fmE5fNYMvTbQBReWLXDYQmBMsthCxES78xVRpamFiBFFamHZAMMsRCowqwUHFLfg7bxCJCjdDLxO0KUVli0osgrLBtpJhUizjyQVriYQSCFSdJJQiJgYgydcxZXJhKuqEpSwcBpRAhKWn40MwxGu+hSJEBgqLIYQMaHJICxboAmESDUQAiGiOwQ/uGoPIw+u+gR0EHk1hUMHEXVF4mDZQDNvECm1H3owa6cNImVmaYOIDQQ2iOhSpEHEwDwsx/ZqQBEyWH53qIgYLBugAYPI68khgMFVu4stWC7u6MB/hiuINHcaGgrOEgWB8yQBFETKn8deTrazBJHXwmoswbK8u8E8oIcRLMvrQQTL+h0IQaSV0pAnBarwwMJZUBMcWC5/GzYQeR9vBuzIBgIDVwM3eIrTAQpEjIykBK4mmgGBSLk76ICImaGOWAYMCCzhHBcQyT1BQgERE3Z0rh4AB4iI+lu8LmgBASJqUX/XpAwCLJtQxACWDQyEAK4W8uiYDFX4X9kAjf5DasGh/xALBPoPkWfJf4gNTexf2QIF/UOqQEP/ECPzaJ87TfxD0iiZ0U9WBuP+VhtuiEuSxvwhJoLWRq4sH4c4Z1jAH2JBD/BX1p/H5/qC4H5ImrJmsh8ieoMPdSjUbzXhx+Zr0kX5FXKIKWL8yuUnIH5I87QQ/BC9ofi+1UYefZUxFtq3mhjF61vltVB95cJToD5gHA2k9K0G4uj4pV5AX1mTxvMhBZ9vcqcxBsy3JtZkmHxI4k6cyYeou+EpWRkiHyLP4PgQ/XYaH6IKo/gQ8TYSH6KoCuIrmwAxfEDp2zF87Vfh4zB8q7y7wWXFQALfakAFvlfYDGuC98olH4rdW20QxD2kd1ncHmIjj7lHUATtFUaPImSvXHotxF5ZnQPsIa0fxkZRa9L1yvrD2Hqrvh5Wr6yvA9Ura2f1fJkdQD0kRbsmUK8wK9AwPaQOfvyyQqH0kDrE4bcimgi9soFh/LxVfxw6bzWQx+aSJZB5rekOhgLzVgtu6DMjjpSHNL4SJq8sHge8MIQJeYg0C8hDbGjS8coWhgUo/x8NUg7IuT8BAA=="
PB_SAMPLE_REF = pd.read_csv(io.BytesIO(gzip.decompress(base64.b64decode(PB_SAMPLE_B64))))
PB_ANSWERS_REF = pd.read_csv(io.BytesIO(gzip.decompress(base64.b64decode(PB_ANSWERS_B64))))

# Verify embedded Part B reference counts: grader 212, finetuned 100, base 99
_ref_right = {r: int((PB_ANSWERS_REF[PB_ANSWERS_REF["run"] == r]["vlm_class"] ==
                      PB_ANSWERS_REF[PB_ANSWERS_REF["run"] == r]["true_class"]).sum())
              for r in ("grader", "finetuned", "base")}
assert _ref_right == {"grader": 212, "finetuned": 100, "base": 99}, _ref_right
print(f"✓ L0b configuration ready{' [SMOKE TEST]' if SMOKE_TEST else ''}; Part B reference verified: {_ref_right}")

✓ L0b configuration ready; Part B reference verified: {'grader': 212, 'finetuned': 100, 'base': 99}


In [3]:
# ── Cell 3: Build the D4 split, verify Part B's 300 validation buildings, and draw 1,800 balanced training crops ──
from sklearn.model_selection import train_test_split

with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
    raw_data = json.load(f)

def name_of(path):
    return Path(str(path).replace("\\", "/")).name

bda_entries = [e for e in raw_data
               if e.get("task") == "Referring Expression Segmentation"
               and e.get("cls_description") == "Building Damage Assessment"]

_listing = {}
def exists(path):
    folder, name = os.path.split(str(path))
    if folder not in _listing:
        _listing[folder] = set(os.listdir(folder)) if os.path.isdir(folder) else set()
    return name in _listing[folder]

def resolve_path(rel_path):
    if not rel_path:
        return None
    rel_path = rel_path.replace("\\", "/")
    filename = Path(rel_path).name
    for c in (DATA_ROOT / rel_path, DATA_ROOT / "train_images" / rel_path,
              DATA_ROOT / "train_images" / "train_images" / filename,
              DATA_ROOT / "masks" / rel_path, DATA_ROOT / "masks" / "masks" / filename):
        if exists(c):
            return str(c)
    return None

by_image = defaultdict(dict)
pre_rel = {}
for e in bda_entries:
    post_img = e.get("post_image_path", "")
    if e.get("image_type") != "Optical":
        continue
    mask_rel = e.get("ground_truth", "")
    folder = Path(mask_rel.replace("\\", "/")).parent.name
    resolved = resolve_path(mask_rel)
    if resolved:
        by_image[post_img][folder] = resolved
        pre_rel[post_img] = e.get("pre_image_path")

FOLDER_TO_CLASS = {
    "train_building_intact_mask": 1,
    "train_building_damaged_mask": 2,
    "train_building_destroyed_mask": 3,
}

pairs = []
for post_img, mask_dict in by_image.items():
    img_path = resolve_path(post_img)
    if img_path and any(folder in FOLDER_TO_CLASS for folder in mask_dict):
        pairs.append({"image_path": img_path, "mask_paths": dict(mask_dict), "pre_rel": pre_rel.get(post_img)})

assert len(pairs) == 6443
train_pairs, val_pairs = train_test_split(pairs, test_size=0.1, random_state=42)
assert len(train_pairs) == 5798 and len(val_pairs) == 645

clean_val_pairs = sorted((p for p in val_pairs if name_of(p["image_path"]) not in LEAK40),
                         key=lambda p: name_of(p["image_path"]))
assert len(clean_val_pairs) == 605
val_path_of = {name_of(p["image_path"]): p["image_path"] for p in clean_val_pairs}

XBD_NAME = re.compile(r"^(.+?)_(\d{8})_post_disaster")
OTHER_NAME = re.compile(r"^(.+?)_post")

def event_and_source(image_name):
    m = XBD_NAME.match(image_name)
    if m:
        return m.group(1), "xBD"
    m = OTHER_NAME.match(image_name)
    return re.sub(r"\d+$", "", m.group(1) if m else Path(image_name).stem), "other"

def true_mask(pair, shape):
    gt = None
    for folder, cls in FOLDER_TO_CLASS.items():
        path = pair["mask_paths"].get(folder)
        m = cv2.imread(path, cv2.IMREAD_GRAYSCALE) if path else None
        if m is None:
            continue
        if gt is None:
            gt = np.zeros_like(m, dtype=np.uint8)
        gt[m > 0] = cls
    if gt is not None and gt.shape != shape:
        gt = cv2.resize(gt, (shape[1], shape[0]), interpolation=cv2.INTER_NEAREST)
    return gt

def pieces(gt):
    out = []
    for cls in (1, 2, 3):
        n, _, stats, _ = cv2.connectedComponentsWithStats((gt == cls).astype(np.uint8), connectivity=8)
        for i in range(1, n):
            x, y, w, h, area = (int(v) for v in stats[i, :5])
            if area >= MIN_AREA:
                out.append({"true_class": CLASSES[cls - 1], "piece_id": i, "x": x, "y": y, "w": w, "h": h, "area_px": area})
    return out

def draw_balanced_sample(pair_list, n_per_class, seed=SEED):
    rows = []
    for pair in pair_list:
        name = name_of(pair["image_path"])
        with Image.open(pair["image_path"]) as im:
            width, height = im.size
        gt = true_mask(pair, (height, width))
        if gt is None:
            continue
        event, source = event_and_source(name)
        for p in pieces(gt):
            rows.append({"image": name, "image_path": pair["image_path"], "event": event, "source": source,
                         "image_w": width, "image_h": height, **p})
    df = pd.DataFrame(rows)
    rng = np.random.default_rng(seed)
    picked, caps_used = [], {}
    for cls in CLASSES:
        cand = df[df["true_class"] == cls].reset_index(drop=True)
        order = rng.permutation(len(cand))
        for cap in CAPS:
            per_image, taken = Counter(), []
            for i in order:
                name = cand.at[i, "image"]
                if cap is None or per_image[name] < cap:
                    taken.append(i)
                    per_image[name] += 1
                if len(taken) == n_per_class:
                    break
            if len(taken) == n_per_class:
                break
        assert len(taken) == n_per_class, f"Only {len(taken)} {cls} buildings found"
        caps_used[cls] = cap
        picked.append(cand.loc[taken])
    sample = pd.concat(picked).sample(frac=1, random_state=seed).reset_index(drop=True)
    sample.insert(0, "item", range(len(sample)))
    return sample, caps_used, df

# 1) Rebuild Part B's 300 validation buildings and check 100% agreement with Tamanna's part_b_sample.csv
val_sample, val_caps, _ = draw_balanced_sample(clean_val_pairs, N_VAL_PER_CLASS, seed=SEED)
check_cols = ["item", "image", "true_class", "piece_id", "x", "y", "w", "h", "area_px"]
assert val_sample[check_cols].equals(PB_SAMPLE_REF[check_cols]), "Rebuilt validation sample differs from Part B's!"
print("✓ Rebuilt validation sample matches Tamanna's part_b_sample.csv on 300 of 300 buildings (100%)")

# 2) Build balanced training sample from training pairs only (shuffling training pairs with seed 42; in smoke test, scan a subset)
train_pool = sorted(train_pairs, key=lambda p: name_of(p["image_path"]))
if SMOKE_TEST:
    train_pool = train_pool[:120]
train_sample, train_caps, _ = draw_balanced_sample(train_pool, N_TRAIN_PER_CLASS, seed=SEED)
assert set(train_sample["image"]).isdisjoint(set(val_sample["image"])), "Leak between training and validation crops!"
print(f"✓ Training sample: {len(train_sample)} buildings ({N_TRAIN_PER_CLASS} per class) from "
      f"{train_sample['image'].nunique()} training images; caps {train_caps}")

# 3) Render and save crops (Part B's exact marked_crop function)
def window_for(x, y, w, h, H, W):
    side = min(max(WINDOW, max(w, h) + 2 * (BOX_PAD + CONTEXT)), H, W)
    cx, cy = x + w / 2, y + h / 2
    left = int(round(min(max(cx - side / 2, 0), W - side)))
    top = int(round(min(max(cy - side / 2, 0), H - side)))
    return top, left, side

def marked_crop(post, x, y, w, h):
    H, W = post.shape[:2]
    top, left, side = window_for(x, y, w, h, H, W)
    scale = OUT_PX / side
    crop = cv2.resize(post[top:top + side, left:left + side], (OUT_PX, OUT_PX),
                      interpolation=cv2.INTER_CUBIC if scale > 1 else cv2.INTER_AREA)
    clamp = lambda v: int(min(max(round(v), 0), OUT_PX - 1))
    p0 = (clamp((x - BOX_PAD - left) * scale), clamp((y - BOX_PAD - top) * scale))
    p1 = (clamp((x + w + BOX_PAD - left) * scale), clamp((y + h + BOX_PAD - top) * scale))
    cv2.rectangle(crop, p0, p1, (255, 0, 0), BOX_THICK)
    return np.ascontiguousarray(crop), side

train_crop_paths = [None] * len(train_sample)
for name, idx in train_sample.groupby("image").groups.items():
    img_path = train_sample.at[idx[0], "image_path"]
    post = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
    for i in idx:
        crop, _ = marked_crop(post, *(int(v) for v in train_sample.loc[i, ["x", "y", "w", "h"]]))
        cpath = CROP_DIR / f"train_{int(train_sample.at[i, 'item']):04d}.png"
        Image.fromarray(crop).save(cpath)
        train_crop_paths[i] = str(cpath)
train_sample["crop_path"] = train_crop_paths
train_sample.to_csv(WORK / "train_crops.csv", index=False)

val_crops = [None] * len(val_sample)
for name, idx in val_sample.groupby("image").groups.items():
    post = cv2.cvtColor(cv2.imread(val_path_of[name]), cv2.COLOR_BGR2RGB)
    for i in idx:
        val_crops[i], _ = marked_crop(post, *(int(v) for v in val_sample.loc[i, ["x", "y", "w", "h"]]))
print(f"✓ Saved {len(train_sample)} training crops to {CROP_DIR} and prepared {len(val_crops)} validation crops")

# Cache base model once before launching 2 DDP processes
base_cache = snapshot_download(START_MODEL, token=os.environ.get("HF_TOKEN"))
CFG = {
    "task": "L0b: vision-encoder + language LoRA on building crops",
    "smoke_test": SMOKE_TEST,
    "start_model": START_MODEL,
    "train_csv": str(WORK / "train_crops.csv"),
    "adapter_dir": str(WORK / "adapter"),
    "class_prompt": CLASS_PROMPT,
    "include_vision_attn": INCLUDE_VISION_ATTN,
    "seed": SEED,
    "lora": {"r": 64, "alpha": 16, "dropout": 0.05},
    "lr": 1e-4, "betas": [0.9, 0.95], "weight_decay": 0.01, "warmup_ratio": 0.05,
    "per_device_batch": 1,
    "grad_accum": 6 if SMOKE_TEST else 16,      # x 2 GPUs = global batch 32 (~56 optimizer steps for 1,800 crops)
    "optim": "adamw_bnb_8bit",
}
(WORK / "config.json").write_text(json.dumps(CFG, indent=1), encoding="utf-8")
print(f"✓ Cached {START_MODEL} at {base_cache} and wrote {WORK / 'config.json'}")

✓ Rebuilt validation sample matches Tamanna's part_b_sample.csv on 300 of 300 buildings (100%)
✓ Training sample: 1800 buildings (600 per class) from 1263 training images; caps {'Intact': 3, 'Damaged': 3, 'Destroyed': 3}
✓ Saved 1800 training crops to /tmp/l0b_crops and prepared 300 validation crops


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

✓ Cached AbrarAlam/disasterm3-qwen2.5vl7b-l0s2-mergedFP at /root/.cache/huggingface/hub/models--AbrarAlam--disasterm3-qwen2.5vl7b-l0s2-mergedFP/snapshots/def8f6c81889ffb68904afdc90c3a1389697d73b and wrote /kaggle/working/L0b/config.json


In [4]:
%%writefile /kaggle/working/L0b/train_l0b.py
# ── Cell 4 (written to train_l0b.py): DDP training with LoRA on BOTH vision blocks and language layers ──
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
import json
import math
import time
from importlib.metadata import version
from pathlib import Path

import pandas as pd
import torch
from PIL import Image
from peft import LoraConfig, get_peft_model
from transformers import (AutoProcessor, BitsAndBytesConfig, Qwen2_5_VLForConditionalGeneration, Trainer,
                          TrainingArguments)

WORK = Path("/kaggle/working/L0b")
CFG = json.loads((WORK / "config.json").read_text(encoding="utf-8"))
RANK, WORLD = int(os.environ.get("LOCAL_RANK", "0")), int(os.environ.get("WORLD_SIZE", "1"))
MAIN = RANK == 0
torch.cuda.set_device(RANK)

def log(msg):
    if MAIN:
        print(msg, flush=True)

df = pd.read_csv(CFG["train_csv"])
examples = [{"crop_path": r.crop_path, "prompt": CFG["class_prompt"], "answer": r.true_class}
            for r in df.itertuples()]
log(f"✓ Training data: {len(examples)} balanced building crops ({df['true_class'].value_counts().to_dict()})")

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    CFG["start_model"], dtype=torch.float16, device_map={"": RANK}, attn_implementation="sdpa",
    quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                           bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True))

# Select LoRA target modules on BOTH language layers AND vision blocks
LLM_PROJ = ("q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj")
VIS_PROJ = ("gate_proj", "up_proj", "down_proj", "qkv", "proj") if CFG["include_vision_attn"] else ("gate_proj", "up_proj", "down_proj")
modules = dict(model.named_modules())
llm_targets = [n for n, m in modules.items() if "visual" not in n and n.rsplit(".", 1)[-1] in LLM_PROJ and hasattr(m, "in_features")]
vis_targets = [n for n, m in modules.items() if "visual.blocks." in n and n.rsplit(".", 1)[-1] in VIS_PROJ and hasattr(m, "in_features")]
assert len(llm_targets) == 28 * 7, f"Expected 196 LLM projections, found {len(llm_targets)}"
assert len(vis_targets) >= 32 * 3, f"Expected >= 96 vision projections, found {len(vis_targets)}"
targets = llm_targets + vis_targets

llm_params = sum(CFG["lora"]["r"] * (modules[n].in_features + modules[n].out_features) for n in llm_targets)
vis_params = sum(CFG["lora"]["r"] * (modules[n].in_features + modules[n].out_features) for n in vis_targets)
model.enable_input_require_grads()
model = get_peft_model(model, LoraConfig(r=CFG["lora"]["r"], lora_alpha=CFG["lora"]["alpha"],
                                         lora_dropout=CFG["lora"]["dropout"], target_modules=targets,
                                         bias="none", task_type="CAUSAL_LM"))

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
vis_lora = [n for n, p in model.named_parameters() if p.requires_grad and "visual" in n]
assert vis_lora and trainable == llm_params + vis_params, f"Mismatch: {trainable} vs {llm_params + vis_params}"
log(f"✓ Model loaded on {WORLD} GPUs: LoRA on {len(llm_targets)} language modules ({llm_params:,} params) "
    f"+ {len(vis_targets)} vision modules ({vis_params:,} params) = {trainable:,} trainable parameters")

processor = AutoProcessor.from_pretrained(CFG["start_model"])
tok = processor.tokenizer
MARKER = tok("<|im_start|>assistant\n", add_special_tokens=False)["input_ids"]
VISION_IDS = torch.tensor([tok.convert_tokens_to_ids(t) for t in ("<|image_pad|>", "<|vision_start|>", "<|vision_end|>")])

def mask_prompt(row):
    ids = row.tolist()
    for start in range(len(ids) - len(MARKER), -1, -1):
        if ids[start:start + len(MARKER)] == MARKER:
            row[:start + len(MARKER)] = -100
            return row
    raise ValueError("No assistant turn in sequence")

def collate(batch):
    texts, images = [], []
    for x in batch:
        img = Image.open(x["crop_path"]).convert("RGB")
        messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": x["prompt"]}]},
                    {"role": "assistant", "content": [{"type": "text", "text": x["answer"]}]}]
        texts.append(processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=False))
        images.append(img)
    out = processor(text=texts, images=images, padding=True, return_tensors="pt")
    labels = out["input_ids"].clone()
    labels[out["attention_mask"] == 0] = -100
    for row in labels:
        mask_prompt(row)
    assert not torch.isin(labels, VISION_IDS).any(), "Vision tokens in loss"
    keep = labels.shape[1] - int((labels != -100).int().argmax(1).min()) + 1
    out["labels"], out["logits_to_keep"] = labels[:, -keep:], keep
    return out

class StepOnCpu(Trainer):
    def get_batch_samples(self, epoch_iterator, *args, **kwargs):
        to_cpu = ({k: (v.to("cpu", torch.float16) if k == "pixel_values" else v.cpu()) if torch.is_tensor(v) else v
                   for k, v in b.items()} for b in epoch_iterator)
        return super().get_batch_samples(to_cpu, *args, **kwargs)

GLOBAL = WORLD * CFG["per_device_batch"] * CFG["grad_accum"]
steps = math.ceil(len(examples) / GLOBAL)
args = TrainingArguments(
    output_dir=str(WORK / "checkpoints"), num_train_epochs=1, per_device_train_batch_size=CFG["per_device_batch"],
    gradient_accumulation_steps=CFG["grad_accum"], learning_rate=CFG["lr"], adam_beta1=CFG["betas"][0],
    adam_beta2=CFG["betas"][1], weight_decay=CFG["weight_decay"], lr_scheduler_type="cosine",
    warmup_steps=max(1, math.ceil(CFG["warmup_ratio"] * steps)), max_grad_norm=1.0, fp16=True,
    gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False},
    logging_steps=1, save_strategy="no", report_to="none", optim=CFG["optim"],
    dataloader_num_workers=2, remove_unused_columns=False, ddp_find_unused_parameters=False, seed=CFG["seed"])

t0 = time.time()
trainer = StepOnCpu(model=model, args=args, train_dataset=examples, data_collator=collate)
log(f"⏳ Training {len(examples)} crops in {steps} optimizer steps (global batch {GLOBAL})")
trainer.train()
train_min = (time.time() - t0) / 60
peak_gb = torch.cuda.max_memory_allocated() / 1e9
print(f"[GPU {RANK}] peak memory {peak_gb:.2f} GB", flush=True)

if MAIN:
    trainer.model.save_pretrained(CFG["adapter_dir"])
    summary = {
        "smoke_test": CFG["smoke_test"], "train_examples": len(examples),
        "llm_lora_modules": len(llm_targets), "llm_lora_params": llm_params,
        "vision_lora_modules": len(vis_targets), "vision_lora_params": vis_params,
        "total_trainable_params": trainable, "global_steps": trainer.state.global_step,
        "train_minutes": round(train_min, 2), "peak_gpu_gb_rank0": round(peak_gb, 2),
        "log_history": trainer.state.log_history,
        "versions": {p: version(p) for p in ("torch", "transformers", "peft", "bitsandbytes", "accelerate")},
    }
    (WORK / "train_summary.json").write_text(json.dumps(summary, indent=1), encoding="utf-8")
    log(f"✓ Saved trained vision+language adapter to {CFG['adapter_dir']} ({train_min:.1f} min, peak {peak_gb:.2f} GB)")

Writing /kaggle/working/L0b/train_l0b.py


In [5]:
# ── Cell 5: Launch DDP training on both T4 GPUs ──
!accelerate launch --multi_gpu --num_processes=2 /kaggle/working/L0b/train_l0b.py
assert (WORK / "train_summary.json").exists(), "Training did not complete; check output above"
TRAIN_SUMMARY = json.loads((WORK / "train_summary.json").read_text(encoding="utf-8"))
losses = [round(e["loss"], 4) for e in TRAIN_SUMMARY["log_history"] if "loss" in e]
print(f"✓ Training complete in {TRAIN_SUMMARY['train_minutes']} min; peak GPU0 {TRAIN_SUMMARY['peak_gpu_gb_rank0']} GB")
print(f"  Losses ({len(losses)} steps): first 3 = {losses[:3]}, last 3 = {losses[-3:]}")

The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_machines` was set to a value of `1`
	`--mixed_precision` was set to a value of `'no'`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
✓ Training data: 1800 balanced building crops ({'Destroyed': 600, 'Damaged': 600, 'Intact': 600})
Loading weights: 100%|████████████████████████| 729/729 [00:52<00:00, 13.94it/s]

✓ Model loaded on 2 GPUs: LoRA on 196 language modules (161,480,704 params) + 160 vision modules (44,605,440 params) = 206,086,144 trainable parameters
⏳ Training 1800 crops in 57 optimizer steps (global batch 32)
  0%|                                                    | 0/57 [00:00<?, ?it/s]/usr/local/lib/python3.13/dist-packages/bitsandbytes/backends/cuda/ops.py:957: UserWarning: inner dimension (3420) is not aligned for fast kernel with blocksize=64, falling back to slower 

In [6]:
# ── Cell 6: Score the vision+language trained 7B on Part B's 300 validation buildings (in fp16, greedy) ──
import gc
from peft import PeftModel
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

def parse_class(answer):
    found = {m.lower() for m in re.findall(r"\b(intact|damaged|destroyed)\b", str(answer), re.IGNORECASE)}
    return found.pop().capitalize() if len(found) == 1 else None

def parse_grade(answer):
    m = re.search(r"grade\s*([1-5])\b", str(answer), re.IGNORECASE) or re.fullmatch(r"\s*([1-5])\s*\.?\s*", str(answer))
    return int(m.group(1)) if m else None

# Load base 7B in fp16 across both T4s (as Part B did) and attach the trained L0b adapter
processor = AutoProcessor.from_pretrained(START_MODEL)
processor.tokenizer.padding_side = "left"
base_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    START_MODEL, dtype=torch.float16, device_map="auto", attn_implementation="sdpa")
model = PeftModel.from_pretrained(base_model, str(WORK / "adapter"))
model.eval()

@torch.inference_mode()
def ask(batch, prompt, max_new_tokens):
    messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": prompt}]}]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text] * len(batch), images=[Image.fromarray(c) for c in batch],
                       padding=True, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, temperature=None,
                         top_p=None, top_k=None, repetition_penalty=1.0)
    return [s.strip() for s in processor.batch_decode(out[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)]

eval_sample = val_sample.iloc[:12].copy() if SMOKE_TEST else val_sample.copy()
eval_crops = val_crops[:len(eval_sample)]
BATCH = 8
t0 = time.time()
class_answers, grade_answers = [], []
for s in range(0, len(eval_crops), BATCH):
    batch = eval_crops[s:s + BATCH]
    class_answers += ask(batch, CLASS_PROMPT, 8)
    grade_answers += ask(batch, GRADE_PROMPT, 12)
eval_sec = time.time() - t0

l0b_rows = []
for r, ca, ga in zip(eval_sample.itertuples(), class_answers, grade_answers):
    grade = parse_grade(ga)
    l0b_rows.append({
        "run": "l0b_vision_lora", "item": int(r.item), "image": r.image, "true_class": r.true_class,
        "class_answer": ca, "vlm_class": parse_class(ca),
        "grade_answer": ga, "vlm_grade": grade, "grade_as_class": GRADE_TO_CLASS.get(grade),
    })
del model, base_model, processor
gc.collect()
torch.cuda.empty_cache()

ref_sub = PB_ANSWERS_REF[PB_ANSWERS_REF["item"].isin(eval_sample["item"])].copy()
answers = pd.concat([ref_sub, pd.DataFrame(l0b_rows)], ignore_index=True)
answers.to_csv(WORK / "l0b_answers.csv", index=False)
print(f"✓ Evaluated {len(eval_sample)} validation buildings in {eval_sec:.1f} s ({eval_sec / len(eval_sample):.2f} s/building)")

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

✓ Evaluated 300 validation buildings in 532.2 s (1.77 s/building)


In [7]:
# ── Cell 7: Confusion matrices, per-class recall, exact McNemar tests, verdict by L0b's rule, and README ──
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.stats import binomtest

RUNS = ["grader", "finetuned", "base", "l0b_vision_lora"]
GRADER_RECORDED = {"Intact": 17878 / (19624 - 601), "Damaged": 2841 / (4370 - 121), "Destroyed": 1031 / (1600 - 132)}

def wilson(k, n, z=1.96):
    if n == 0:
        return float("nan"), float("nan")
    p, z2 = k / n, z * z
    centre = (p + z2 / (2 * n)) / (1 + z2 / n)
    half = z * np.sqrt(p * (1 - p) / n + z2 / (4 * n * n)) / (1 + z2 / n)
    return centre - half, centre + half

def mcnemar(only_a, only_b):
    n = only_a + only_b
    return 1.0 if n == 0 else float(binomtest(only_a, n, 0.5).pvalue)

boot_rng = np.random.default_rng(SEED)
recall_rows, cm_rows = [], []
for run in RUNS:
    sub = answers[answers["run"] == run]
    cm = (pd.crosstab(sub["true_class"], sub["vlm_class"].fillna("unparsed"))
          .reindex(index=CLASSES, columns=CLASSES + ["unparsed"], fill_value=0))
    print(f"\n{run} confusion matrix (rows = true class, columns = predicted):")
    print(cm.to_string())
    for cls in CLASSES:
        for col in CLASSES + ["unparsed"]:
            cm_rows.append({"run": run, "true_class": cls, "answer": col, "buildings": int(cm.at[cls, col])})
    outcomes = {}
    for cls in CLASSES:
        n = int(cm.loc[cls].sum())
        parsed = n - int(cm.at[cls, "unparsed"])
        right = int(cm.at[cls, cls])
        lo, hi = wilson(right, parsed)
        outcomes[cls] = (sub[(sub["true_class"] == cls) & sub["vlm_class"].notna()]["vlm_class"] == cls).to_numpy()
        both = sub[(sub["true_class"] == cls) & sub["vlm_class"].notna() & sub["grade_as_class"].notna()]
        recall_rows.append({
            "run": run, "class": cls, "buildings": n, "parsed": parsed, "right": right,
            "recall": round(right / parsed, 4) if parsed else None, "ci_low": round(lo, 4), "ci_high": round(hi, 4),
            "class_matches_own_grade": "" if run == "grader" else f"{int((both['vlm_class'] == both['grade_as_class']).sum())}/{len(both)}",
        })
    if all(len(o) for o in outcomes.values()):
        mean_rec = float(np.mean([o.mean() for o in outcomes.values()]))
        boots = np.mean([[boot_rng.choice(o, len(o)).mean() for o in outcomes.values()] for _ in range(2000)], axis=1)
        lo, hi = np.percentile(boots, [2.5, 97.5])
    else:
        mean_rec, lo, hi = float("nan"), float("nan"), float("nan")
    recall_rows.append({
        "run": run, "class": "mean of the three", "buildings": len(sub),
        "parsed": int(sub["vlm_class"].notna().sum()), "right": int((sub["vlm_class"] == sub["true_class"]).sum()),
        "recall": round(mean_rec, 4), "ci_low": round(lo, 4), "ci_high": round(hi, 4), "class_matches_own_grade": "",
    })

recall_df = pd.DataFrame(recall_rows)
cm_df = pd.DataFrame(cm_rows)
recall_df.to_csv(WORK / "l0b_recall.csv", index=False)
cm_df.to_csv(WORK / "l0b_confusion.csv", index=False)

# Paired exact McNemar tests for l0b_vision_lora against grader, finetuned (frozen vision), and base
right = (answers.assign(right=answers["vlm_class"] == answers["true_class"])
         .pivot(index="item", columns="run", values="right")
         .join(eval_sample.set_index("item")["true_class"]))
paired_rows = []
for other in ("grader", "finetuned", "base"):
    for cls in CLASSES + ["all three"]:
        w = right if cls == "all three" else right[right["true_class"] == cls]
        ra, rb = w["l0b_vision_lora"].astype(bool), w[other].astype(bool)
        only_a, only_b = int((ra & ~rb).sum()), int((~ra & rb).sum())
        p = mcnemar(only_a, only_b)
        paired_rows.append({
            "comparison": f"l0b_vision_lora vs {other}", "class": cls, "buildings": len(w),
            "l0b_right": int(ra.sum()), "other_right": int(rb.sum()),
            "both_right": int((ra & rb).sum()), "only_l0b_right": only_a, "only_other_right": only_b,
            "both_wrong": int((~ra & ~rb).sum()), "mcnemar_p": round(p, 6),
            "verdict": ("l0b higher" if p < 0.05 and only_a > only_b else
                        f"l0b lower than {other}" if p < 0.05 and only_a < only_b else
                        "not distinguishable"),
        })
paired_df = pd.DataFrame(paired_rows)
paired_df.to_csv(WORK / "l0b_paired.csv", index=False)
print("\n✓ Paired exact McNemar tests:")
print(paired_df.to_string(index=False))

# Apply Roadmap L0b's decision rule on all three classes combined
vs_grader = paired_df[(paired_df["comparison"] == "l0b_vision_lora vs grader") & (paired_df["class"] == "all three")].iloc[0]
vs_ft = paired_df[(paired_df["comparison"] == "l0b_vision_lora vs finetuned") & (paired_df["class"] == "all three")].iloc[0]
l0b_total = int(vs_grader["l0b_right"])
if not (vs_grader["mcnemar_p"] < 0.05 and vs_grader["only_l0b_right"] < vs_grader["only_other_right"]):
    rule_verdict = (f"Worth a full run: l0b_vision_lora ({l0b_total}/{len(eval_sample)}) is not clearly below the grader "
                    f"({vs_grader['other_right']}/{len(eval_sample)}, McNemar p = {vs_grader['mcnemar_p']}).")
elif vs_ft["mcnemar_p"] < 0.05 and vs_ft["only_l0b_right"] > vs_ft["only_other_right"]:
    rule_verdict = (f"Beats the frozen-vision 7B ({l0b_total} vs {vs_ft['other_right']} of {len(eval_sample)}, "
                    f"only_l0b={vs_ft['only_l0b_right']} vs only_ft={vs_ft['only_other_right']}, McNemar p = {vs_ft['mcnemar_p']}), "
                    f"but stays clearly below the Siamese grader ({l0b_total} vs {vs_grader['other_right']}, "
                    f"only_l0b={vs_grader['only_l0b_right']} vs only_grader={vs_grader['only_other_right']}, p = {vs_grader['mcnemar_p']}). "
                    "By L0b's rule: record that training the vision encoder helps the 7B see damage, and stop — "
                    "the 45 MB grader keeps the grading, and a 34-GPU-hour full run is not needed.")
else:
    rule_verdict = (f"Does not clearly beat the frozen-vision 7B ({l0b_total} vs {vs_ft['other_right']} of {len(eval_sample)}, "
                    f"McNemar p = {vs_ft['mcnemar_p']}). The vision encoder stays frozen.")
print(f"\nVerdict: {rule_verdict}")

# Recall bar chart
x = np.arange(len(CLASSES))
fig, ax = plt.subplots(figsize=(10.5, 4.8), dpi=120)
colors = ["#616161", "#c62828", "#1976d2", "#2e7d32"]
labels_map = {"grader": "Siamese grader", "finetuned": "frozen-vision 7B (L0)",
              "base": "base 7B", "l0b_vision_lora": "vision+language LoRA 7B (L0b)"}
for k, (run, col) in enumerate(zip(RUNS, colors)):
    r = recall_df[(recall_df["run"] == run) & recall_df["class"].isin(CLASSES)].set_index("class").reindex(CLASSES)
    err = np.vstack([r["recall"] - r["ci_low"], r["ci_high"] - r["recall"]]).astype(float)
    ax.bar(x + (k - 1.5) * 0.2, r["recall"].astype(float), width=0.2, color=col, yerr=err, capsize=3, label=labels_map[run])
ax.set_xticks(x, CLASSES)
ax.set_ylim(0, 1.05)
ax.set_ylabel("Recall (parsed answers)")
ax.set_title(f"L0b: {len(eval_sample)} true validation buildings, 95% Wilson intervals")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.08), ncol=4, frameon=False, fontsize=8)
fig.tight_layout()
fig.savefig(WORK / "l0b_recall_chart.png", bbox_inches="tight")
plt.close(fig)

def as_md(df):
    try:
        return df.to_markdown(index=False)
    except ImportError:
        return "```\n" + df.to_string(index=False) + "\n```"

(WORK / "README.md").write_text("\n".join([
    f"# L0b results: training the 7B's vision encoder on building crops{' (SMOKE TEST)' if SMOKE_TEST else ''}", "",
    f"Start model `{START_MODEL}` (4-bit NF4), LoRA (`r=64, alpha=16`) on {TRAIN_SUMMARY['llm_lora_modules']} language "
    f"modules ({TRAIN_SUMMARY['llm_lora_params']:,} params) + {TRAIN_SUMMARY['vision_lora_modules']} vision modules "
    f"({TRAIN_SUMMARY['vision_lora_params']:,} params) = {TRAIN_SUMMARY['total_trainable_params']:,} trainable parameters. "
    f"Trained on {TRAIN_SUMMARY['train_examples']} balanced training building crops in {TRAIN_SUMMARY['global_steps']} steps "
    f"({TRAIN_SUMMARY['train_minutes']} min on 2x T4, peak GPU0 {TRAIN_SUMMARY['peak_gpu_gb_rank0']} GB).", "",
    "## Recall on Part B's 300 true validation buildings (100 per class, 605 clean pairs)", "",
    as_md(recall_df), "",
    "## Paired exact McNemar tests", "",
    as_md(paired_df), "",
    "## Verdict (by the rule fixed in Roadmap L0b)", "",
    rule_verdict, "",
    "![Recall chart](l0b_recall_chart.png)", "",
    "## Files", "",
    "- `config.json`, `train_summary.json`, `l0b_answers.csv`, `l0b_confusion.csv`, `l0b_recall.csv`, `l0b_paired.csv`, `l0b_recall_chart.png`",
]) + "\n", encoding="utf-8")
print(f"✓ Wrote README.md and all L0b result files to {WORK}")


grader confusion matrix (rows = true class, columns = predicted):
vlm_class   Intact  Damaged  Destroyed  unparsed
true_class                                      
Intact          92        4          3         1
Damaged         35       54          5         6
Destroyed        8       17         66         9

finetuned confusion matrix (rows = true class, columns = predicted):
vlm_class   Intact  Damaged  Destroyed  unparsed
true_class                                      
Intact         100        0          0         0
Damaged        100        0          0         0
Destroyed       98        2          0         0

base confusion matrix (rows = true class, columns = predicted):
vlm_class   Intact  Damaged  Destroyed  unparsed
true_class                                      
Intact          49       51          0         0
Damaged         50       50          0         0
Destroyed       38       62          0         0

l0b_vision_lora confusion matrix (rows = true class, columns =